# 01_shared_data_cleaning

This notebook standardizes the five raw project datasets and produces the shared cleaned tables used by both the benchmark and realistic models.

Inputs:
- regulated childcare facilities
- potential locations
- employment rates by ZIP code
- average individual income by ZIP code
- child population by ZIP code

Outputs:
- cleaned facility, demographic, and candidate-site tables
- a master ZIP-code universe for downstream modeling
- geo-ready tables for later distance screening


In [1]:
# 01_shared_data_cleaning.ipynb
# Purpose:
# 1) Load all raw datasets
# 2) Standardize column names and zipcode formats
# 3) Create clean shared datasets for both ideal and realistic models
# 4) Build master zipcode universe
# 5) Export geo-ready facility and candidate tables

## Setup and project paths

This opening block imports packages, defines display settings, and points the notebook to the raw input files used throughout the pipeline.


In [2]:
import os
from pathlib import Path
import pandas as pd
import numpy as np

In [3]:
# Display options
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)
pd.set_option("display.max_rows", 200)

In [4]:
# Project paths
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_SHARED_DIR = PROJECT_ROOT / "data" / "processed" / "shared"

PROCESSED_SHARED_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw data dir:", RAW_DIR)
print("Processed shared dir:", PROCESSED_SHARED_DIR)

Project root: /Users/alexandresepulvedadedietrich/Documents/School/Columbia/Spring_2026/Optimization/group_project
Raw data dir: /Users/alexandresepulvedadedietrich/Documents/School/Columbia/Spring_2026/Optimization/group_project/data/raw
Processed shared dir: /Users/alexandresepulvedadedietrich/Documents/School/Columbia/Spring_2026/Optimization/group_project/data/processed/shared


In [5]:
#Load raw datasets
cc = pd.read_csv(RAW_DIR / "child_care_regulated_nyc.csv")
pop = pd.read_csv(RAW_DIR / "population_nyc.csv")
emp = pd.read_csv(RAW_DIR / "employment_rate_nyc.csv")
inc = pd.read_csv(RAW_DIR / "avg_individual_income_nyc.csv")
pot = pd.read_csv(RAW_DIR / "potential_locations_nyc.csv")

In [6]:
#Quick inspection
datasets = {
    "child_care": cc,
    "population": pop,
    "employment": emp,
    "income": inc,
    "potential_locations": pot
}

for name, df in datasets.items():
    print("=" * 80)
    print(name)
    print("shape:", df.shape)
    print("columns:", df.columns.tolist())
    print(df.head(2))

child_care
shape: (7740, 16)
columns: ['Unnamed: 0', 'facility_id', 'program_type', 'facility_status', 'facility_name', 'city', 'zipcode', 'school_district_name', 'infant_capacity', 'toddler_capacity', 'preschool_capacity', 'school_age_capacity', 'children_capacity', 'total_capacity', 'latitude', 'longitude']
   Unnamed: 0  facility_id program_type facility_status       facility_name      city  zipcode school_district_name  infant_capacity  \
0           0        51349          FDC    Registration     Valerio, Gladys     Bronx    10474              Bronx 8                0   
1           1        73544         SACC    Registration  YMCA OF GREATER NY  New York    10017          Manhattan 2                0   

   toddler_capacity  preschool_capacity  school_age_capacity  children_capacity  total_capacity   latitude  longitude  
0                 0                   0                    0                  6               6  40.818399 -73.888816  
1                 0                   0 

## Cleaning helpers and standardization rules

These steps define the reusable cleaning rules applied across datasets: dropping unnamed columns, standardizing ZIP codes, harmonizing column names, and converting numeric fields.


In [7]:
#Helper functions
def standardize_zipcode(series):
    """
    Convert zipcode to 5-digit string.
    """
    return (
        series.astype(str)
              .str.strip()
              .str.replace(r"\.0$", "", regex=True)
              .str.zfill(5)
    )

def clean_unnamed_columns(df):
    """
    Drop unnamed index-like columns.
    """
    return df.loc[:, ~df.columns.str.contains("^Unnamed")].copy()

def make_numeric(df, cols):
    """
    Convert selected columns to numeric.
    """
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

In [8]:
#Drop unnamed columns
cc  = clean_unnamed_columns(cc)
pop = clean_unnamed_columns(pop)
emp = clean_unnamed_columns(emp)
inc = clean_unnamed_columns(inc)
pot = clean_unnamed_columns(pot)

In [9]:
#Standardize zipcodes
for df in [cc, pop, emp, inc, pot]:
    df["zipcode"] = standardize_zipcode(df["zipcode"])

In [10]:
#Standardize column names
emp = emp.rename(columns={"employment rate": "employment_rate"})
inc = inc.rename(columns={"average income": "average_income"})
pop = pop.rename(columns={"-5": "pop_0_5", "6-12": "pop_6_12", "13-14": "pop_13_14"})

In [11]:
#Convert numeric columns
capacity_cols = [
    "infant_capacity",
    "toddler_capacity",
    "preschool_capacity",
    "school_age_capacity",
    "children_capacity",
    "total_capacity",
    "latitude",
    "longitude"
]

population_numeric_cols = [
    "Total", "pop_0_5", "pop_6_12", "pop_13_14",
    "15-19", "20-24", "25-29", "30-34", "35-39", "40-44",
    "45-49", "50-54", "55-59", "60-64", "65-69", "70-74",
    "75-79", "80-84", "85+"
]

cc = make_numeric(cc, capacity_cols)
pop = make_numeric(pop, population_numeric_cols)
emp = make_numeric(emp, ["employment_rate"])
inc = make_numeric(inc, ["average_income"])
pot = make_numeric(pot, ["latitude", "longitude"])

In [12]:
#Missingness summary
def missing_summary(df, name):
    summary = pd.DataFrame({
        "column": df.columns,
        "missing_count": df.isna().sum().values,
        "missing_pct": (df.isna().mean().values * 100).round(2)
    }).sort_values(by="missing_pct", ascending=False)
    print(f"\nMissingness summary: {name}")
    display(summary)

missing_summary(cc, "child_care")
missing_summary(pop, "population")
missing_summary(emp, "employment")
missing_summary(inc, "income")
missing_summary(pot, "potential_locations")


Missingness summary: child_care


,column,missing_count,missing_pct
13,latitude,169,2.18
14,longitude,169,2.18
6,school_district_name,15,0.19
0,facility_id,0,0.00
1,program_type,0,0.00
2,facility_status,0,0.00
3,facility_name,0,0.00
4,city,0,0.00
5,zipcode,0,0.00
7,infant_capacity,0,0.00



Missingness summary: population


,column,missing_count,missing_pct
0,zipcode,0,0.0
1,Total,0,0.0
18,80-84,0,0.0
17,75-79,0,0.0
16,70-74,0,0.0
15,65-69,0,0.0
14,60-64,0,0.0
13,55-59,0,0.0
12,50-54,0,0.0
11,45-49,0,0.0



Missingness summary: employment


,column,missing_count,missing_pct
0,zipcode,0,0.0
1,employment_rate,0,0.0



Missingness summary: income


,column,missing_count,missing_pct
0,zipcode,0,0.0
1,average_income,0,0.0



Missingness summary: potential_locations


,column,missing_count,missing_pct
0,zipcode,0,0.0
1,latitude,0,0.0
2,longitude,0,0.0


In [13]:
#Add useful flags to childcare table
active_statuses = {"License", "Registration"}

cc["is_active_facility"] = cc["facility_status"].isin(active_statuses).astype(int)
cc["geo_missing_flag"] = cc["latitude"].isna() | cc["longitude"].isna()

# Under-5 current capacity proxy:
# Based on instructor clarification, children_capacity can be treated as the general
# younger-child capacity bucket when age subcolumns are not fully available.
cc["under5_capacity_current"] = cc["children_capacity"]

# Sanity flags
cc["capacity_inconsistency_flag"] = (cc["children_capacity"] > cc["total_capacity"]).astype(int)
cc["zero_total_capacity_flag"] = (cc["total_capacity"].fillna(0) <= 0).astype(int)

In [14]:
#Optional sanity checks
print("Duplicate facility_id count:", cc["facility_id"].duplicated().sum())
print("Facilities with missing coordinates:", cc["geo_missing_flag"].sum())
print("Facilities marked active:", cc["is_active_facility"].sum())
print("Facilities with children_capacity > total_capacity:", cc["capacity_inconsistency_flag"].sum())

Duplicate facility_id count: 0
Facilities with missing coordinates: 169
Facilities marked active: 7713
Facilities with children_capacity > total_capacity: 0


## Dataset-specific cleaned tables

After the common cleaning logic is in place, the notebook constructs cleaned versions of the population, employment, income, facility, and potential-location tables.


In [15]:
#Build cleaned population table
pop_clean = pop.copy()

# Keep a general child population column that can be reused later
pop_clean["child_pop_0_12"] = pop_clean["pop_0_5"] + pop_clean["pop_6_12"]

# Optional approximation for ages 0-12 if you later want to use part of 13-14 in sensitivity
pop_clean["child_pop_0_12_plus_partial_13_14"] = (
    pop_clean["pop_0_5"] + pop_clean["pop_6_12"]
)

In [16]:
#Build clean employment and income tables
emp_clean = emp[["zipcode", "employment_rate"]].copy()
inc_clean = inc[["zipcode", "average_income"]].copy()

In [17]:
#Build cleaned potential locations table
pot_clean = pot.copy()
pot_clean["candidate_id"] = np.arange(1, len(pot_clean) + 1)
pot_clean["geo_missing_flag"] = pot_clean["latitude"].isna() | pot_clean["longitude"].isna()

## Master ZIP universe and geo-ready outputs

This section merges the cleaned tables into a common ZIP-code frame and prepares the geo-ready files used later in the realistic siting model.


In [18]:
#Build master zipcode universe
all_zipcodes = sorted(
    set(cc["zipcode"].dropna().unique())
    | set(pop_clean["zipcode"].dropna().unique())
    | set(emp_clean["zipcode"].dropna().unique())
    | set(inc_clean["zipcode"].dropna().unique())
    | set(pot_clean["zipcode"].dropna().unique())
)

master_zip = pd.DataFrame({"zipcode": all_zipcodes})

master_zip["in_childcare"] = master_zip["zipcode"].isin(cc["zipcode"]).astype(int)
master_zip["in_population"] = master_zip["zipcode"].isin(pop_clean["zipcode"]).astype(int)
master_zip["in_employment"] = master_zip["zipcode"].isin(emp_clean["zipcode"]).astype(int)
master_zip["in_income"] = master_zip["zipcode"].isin(inc_clean["zipcode"]).astype(int)
master_zip["in_potential_locations"] = master_zip["zipcode"].isin(pot_clean["zipcode"]).astype(int)

In [19]:
#Add zipcode-level descriptive summaries
facility_zip_summary = (
    cc.groupby("zipcode")
      .agg(
          n_facilities=("facility_id", "count"),
          n_active_facilities=("is_active_facility", "sum"),
          existing_total_capacity=("total_capacity", "sum"),
          existing_under5_capacity=("under5_capacity_current", "sum"),
          n_missing_geo=("geo_missing_flag", "sum")
      )
      .reset_index()
)

candidate_zip_summary = (
    pot_clean.groupby("zipcode")
             .agg(
                 n_candidate_points=("candidate_id", "count")
             )
             .reset_index()
)

master_zip = master_zip.merge(facility_zip_summary, on="zipcode", how="left")
master_zip = master_zip.merge(candidate_zip_summary, on="zipcode", how="left")

fill_zero_cols = [
    "n_facilities",
    "n_active_facilities",
    "existing_total_capacity",
    "existing_under5_capacity",
    "n_missing_geo",
    "n_candidate_points"
]

for c in fill_zero_cols:
    if c in master_zip.columns:
        master_zip[c] = master_zip[c].fillna(0)

In [20]:
#Create geo-ready tables
facility_geo_ready = cc[[
    "facility_id",
    "facility_name",
    "program_type",
    "facility_status",
    "is_active_facility",
    "zipcode",
    "latitude",
    "longitude",
    "geo_missing_flag",
    "infant_capacity",
    "toddler_capacity",
    "preschool_capacity",
    "school_age_capacity",
    "children_capacity",
    "under5_capacity_current",
    "total_capacity"
]].copy()

potential_locations_geo_ready = pot_clean[[
    "candidate_id",
    "zipcode",
    "latitude",
    "longitude",
    "geo_missing_flag"
]].copy()

In [21]:
#Final shared cleaned tables
clean_childcare_facilities = cc.copy()
clean_population = pop_clean.copy()
clean_employment = emp_clean.copy()
clean_income = inc_clean.copy()
clean_potential_locations = pot_clean.copy()

## Exports and quality checks

The final cells write the shared cleaned datasets to disk and display a compact QA preview so later notebooks can rely on these exports as the common source of truth.


In [22]:
#Save outputs
clean_childcare_facilities.to_csv(PROCESSED_SHARED_DIR / "clean_childcare_facilities.csv", index=False)
clean_population.to_csv(PROCESSED_SHARED_DIR / "clean_population.csv", index=False)
clean_employment.to_csv(PROCESSED_SHARED_DIR / "clean_employment.csv", index=False)
clean_income.to_csv(PROCESSED_SHARED_DIR / "clean_income.csv", index=False)
clean_potential_locations.to_csv(PROCESSED_SHARED_DIR / "clean_potential_locations.csv", index=False)

master_zip.to_csv(PROCESSED_SHARED_DIR / "master_zipcodes.csv", index=False)
facility_geo_ready.to_csv(PROCESSED_SHARED_DIR / "facility_geo_ready.csv", index=False)
potential_locations_geo_ready.to_csv(PROCESSED_SHARED_DIR / "potential_locations_geo_ready.csv", index=False)

print("Shared cleaned files saved to:", PROCESSED_SHARED_DIR)

Shared cleaned files saved to: /Users/alexandresepulvedadedietrich/Documents/School/Columbia/Spring_2026/Optimization/group_project/data/processed/shared


In [23]:
#Final QA preview
print("clean_childcare_facilities:", clean_childcare_facilities.shape)
print("clean_population:", clean_population.shape)
print("clean_employment:", clean_employment.shape)
print("clean_income:", clean_income.shape)
print("clean_potential_locations:", clean_potential_locations.shape)
print("master_zipcodes:", master_zip.shape)
print("facility_geo_ready:", facility_geo_ready.shape)
print("potential_locations_geo_ready:", potential_locations_geo_ready.shape)

display(master_zip.head())
display(clean_childcare_facilities.head())
display(clean_potential_locations.head())

clean_childcare_facilities: (7740, 20)
clean_population: (211, 22)
clean_employment: (179, 2)
clean_income: (179, 2)
clean_potential_locations: (31100, 5)
master_zipcodes: (311, 12)
facility_geo_ready: (7740, 16)
potential_locations_geo_ready: (31100, 5)


,zipcode,in_childcare,in_population,in_employment,in_income,in_potential_locations,n_facilities,n_active_facilities,existing_total_capacity,existing_under5_capacity,n_missing_geo,n_candidate_points
0,10001,1,1,1,1,1,9.0,9.0,609.0,24.0,0.0,100
1,10002,1,1,1,1,1,57.0,56.0,4729.0,203.0,1.0,100
2,10003,1,1,1,1,1,7.0,7.0,1995.0,0.0,0.0,100
3,10004,1,1,1,1,1,2.0,2.0,263.0,0.0,0.0,100
4,10005,1,1,1,1,1,1.0,1.0,39.0,0.0,0.0,100


,facility_id,program_type,facility_status,facility_name,city,zipcode,school_district_name,infant_capacity,toddler_capacity,preschool_capacity,school_age_capacity,children_capacity,total_capacity,latitude,longitude,is_active_facility,geo_missing_flag,under5_capacity_current,capacity_inconsistency_flag,zero_total_capacity_flag
0,51349,FDC,Registration,"Valerio, Gladys",Bronx,10474,Bronx 8,0,0,0,0,6,6,40.818399,-73.888816,1,False,6,0,0
1,73544,SACC,Registration,YMCA OF GREATER NY,New York,10017,Manhattan 2,0,0,0,60,0,60,40.753216,-73.970794,1,False,0,0,0
2,108407,GFDC,License,"Almond Tree Group Family Day Care, LLC",Brooklyn,11225,Brooklyn 17,0,0,0,4,12,16,NaN,NaN,1,True,12,0,0
3,111953,GFDC,License,"Williams, Petal",Brooklyn,11226,Brooklyn 22,0,0,0,4,12,16,NaN,NaN,1,True,12,0,0
4,126425,GFDC,License,"Hernandez, Lidia",Bronx,10465,Bronx 8,0,0,0,0,12,12,40.841228,-73.823572,1,False,12,0,0


,zipcode,latitude,longitude,candidate_id,geo_missing_flag
0,10001,40.741893,-74.000140,1,False
1,10001,40.752007,-74.005436,2,False
2,10001,40.750545,-73.997147,3,False
3,10001,40.744080,-74.001932,4,False
4,10001,40.748690,-73.999341,5,False
